# import library

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

# data processing

In [ ]:
spambase = fetch_ucirepo(id=94)


In [9]:
X = spambase.data.features
y = spambase.data.targets.iloc[:, 0]  # Series nhãn (1 = spam, 0 = ham)

# 2. Kiểm tra điều kiện đầu vào trên X
assert X.shape == (4601, 57), f"Sai shape: nhận được {X.shape}, mong đợi (4601, 57)"
assert int(X.isna().sum().sum()) == 0, "Phát hiện giá trị thiếu trong X"
assert int((X < 0).sum().sum()) == 0, "Phát hiện giá trị âm trong X"
assert len(X) == len(y), "Số dòng giữa X và y không khớp"

# 3. Đếm số dòng trùng lặp hoàn toàn dựa trên 57 cột đặc trưng
dup_mask = X.duplicated(keep="first")  # True ở các dòng trùng lặp cần loại bỏ
dup_count = int(dup_mask.sum())
print(f"Số dòng trùng lặp hoàn toàn (dựa trên 57 đặc trưng): {dup_count}")

# 4. Loại bỏ dòng trùng trên X và loại bỏ đúng các dòng target tương ứng trên y
X_clean = X[~dup_mask].reset_index(drop=True)
y_clean = y[~dup_mask].reset_index(drop=True)

# 5. Kiểm tra lại sau khi loại trùng
assert X_clean.duplicated().sum() == 0, "Vẫn còn dòng trùng lặp trong X_clean!"
assert len(X_clean) == len(y_clean), "Sau khi lọc, số dòng của X_clean và y_clean không khớp!"

print(f"Sau khi lọc: X_clean = {X_clean.shape}, y_clean = {y_clean.shape}")
print("Phân bố nhãn sau khi lọc (0=ham, 1=spam):")
print(y_clean.value_counts(normalize=True).round(4))

Số dòng trùng lặp hoàn toàn (dựa trên 57 đặc trưng): 394
Sau khi lọc: X_clean = (4207, 57), y_clean = (4207,)
Phân bố nhãn sau khi lọc (0=ham, 1=spam):
Class
0    0.6009
1    0.3991
Name: proportion, dtype: float64


In [13]:
X_clean.head(3)

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,word_freq_conference,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.0,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.0,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259


# Splitting data for training, validation, and test.

In [15]:
class DatasetSplitter:
    """Thực hiện tách tập test ngay khi khởi tạo, giữ test khóa cho đến khi mở 1 lần."""

    def __init__(self, X, y, test_size=0.20, random_state=42):
        X_temp, X_test, y_temp, y_test = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state,
            stratify=y,
        )
        self._X_temp = X_temp
        self._y_temp = y_temp
        self._X_test = X_test
        self._y_test = y_test
        self._test_opened = False

    def get_temp(self):
        """Lấy phần dữ liệu còn lại (80%) để chia tiếp train và validation."""
        return self._X_temp, self._y_temp

    def open_test(self):
        """Mở tập test đúng 1 lần duy nhất khi cần đánh giá cuối cùng."""
        if self._test_opened:
            raise RuntimeError(
                "Tập test đã được mở trước đó! Không được truy cập lại."
            )
        self._test_opened = True
        return self._X_test, self._y_test


# 1. Khởi tạo đối tượng và thực hiện chia lần 1 ngay bên trong class
splitter = DatasetSplitter(X_clean, y_clean, test_size=0.20, random_state=42)

# 2. Lấy phần còn lại (temp) ra để chia tiếp train (60%) và validation (20%)
X_temp, y_temp = splitter.get_temp()
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp,
)

# --- KIỂM TRA (ASSERT) ---

# Mở tập test thông qua đối tượng splitter (chỉ mở 1 lần để kiểm tra/tổng kết)
X_test, y_test = splitter.open_test()

# Xác nhận gọi open_test() lần thứ 2 sẽ báo lỗi
try:
    splitter.open_test()
    raise AssertionError("Lần gọi open_test() thứ 2 lẽ ra phải báo lỗi!")
except RuntimeError:
    pass

# 1. Tổng kích thước 3 tập phải khớp với X_clean / y_clean
total_len = len(X_clean)
assert len(X_train) + len(X_val) + len(X_test) == total_len, (
    "Tổng số mẫu 3 tập không khớp với dữ liệu gốc!"
)
assert (
    len(X_train) == len(y_train)
    and len(X_val) == len(y_val)
    and len(X_test) == len(y_test)
), "Số lượng X và y không khớp ở ít nhất một tập!"

# 2. Không có email nào xuất hiện trong hai tập (kiểm tra qua index gốc sau reset_index)
train_idx = set(X_train.index)
val_idx = set(X_val.index)
test_idx = set(X_test.index)

assert train_idx.isdisjoint(val_idx), "Trùng lặp mẫu giữa train và validation!"
assert train_idx.isdisjoint(test_idx), "Trùng lặp mẫu giữa train và test!"
assert val_idx.isdisjoint(test_idx), "Trùng lặp mẫu giữa validation và test!"

# 3. Mỗi tập giữ tỷ lệ spam xấp xỉ nhau (sai số tối đa 2% so với tỷ lệ gốc)
base_spam_rate = float(y_clean.mean())
for name, y_split in [("train", y_train), ("val", y_val), ("test", y_test)]:
    rate = float(y_split.mean())
    assert abs(rate - base_spam_rate) < 0.02, (
        f"Tỷ lệ spam ở tập {name} ({rate:.4f}) lệch quá nhiều so với gốc ({base_spam_rate:.4f})"
    )

# In thông tin tổng kết
for name, X_s, y_s in [
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test),
]:
    pct = len(X_s) / total_len * 100
    print(
        f"{name:10s}: shape={X_s.shape} ({pct:.1f}%) | tỷ lệ spam={y_s.mean():.4f}"
    )

Train     : shape=(2523, 57) (60.0%) | tỷ lệ spam=0.3991
Validation: shape=(842, 57) (20.0%) | tỷ lệ spam=0.3990
Test      : shape=(842, 57) (20.0%) | tỷ lệ spam=0.3990


# Check the same division method on the original file.

In [16]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp,
)

# 3. Đếm số email trong X_val có bản sao y hệt trong X_train (dựa trên 57 đặc trưng)
train_tuples = set(X_train.itertuples(index=False, name=None))
leak_count = sum(
    row in train_tuples
    for row in X_val.itertuples(index=False, name=None)
)

print(f"leak_count: {leak_count} / {len(X_val)} email trong validation có bản sao y hệt trong train")

leak_count: 83 / 920 email trong validation có bản sao y hệt trong train
